In [20]:
from datasets import load_dataset
import json
import numpy as np
import random


In [21]:
system_prompt = """You are an event extraction system. 
Your task is to assign the correct event type and argument roles to a given trigger and its arguments.
You will be provided with: an input text, a trigger span

IMPORTANT: Output ONLY valid JSON. Output Format (JSON only, no markdown):
{"events": [[<trigger span>, <event type>], [<trigger span 2>, <event type 2>]]}
"""

user_prompt = """
Input text:
{text}

Trigger:
{trigger}
"""

In [22]:
t_system_prompt = """You are an event extraction system. 
Your task is to assign the correct event type and argument roles to a given trigger and its arguments.
You will be provided with: an input text, a trigger span

IMPORTANT: Output ONLY valid JSON. Output Format (JSON only, no markdown):
{"events": [[<trigger span>, <event type>], [<trigger span 2>, <event type 2>]]}
"""

t_user_prompt = """
Input text:
{text}

Trigger:
{trigger}

Here is a reference response to the input sentence:
{result}
You may include additional valid triggers not in the reference. Now provide your own extraction, including the thinking process::
"""

In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B', padding_side="right")

In [24]:
def extract_unique_labels(dataset):
    event_types = set()
    argument_roles = set()
    
    for sample in dataset:
        events = sample.get('events', [])
        
        for event in events:
            if 'type' in event:
                event_types.add(event['type'])
                
            mentions = event.get('mention', [])
            for mention in mentions:
                arguments = mention.get('arguments', [])
                
                for arg in arguments:
                    if 'role' in arg:
                        argument_roles.add(arg['role'])
                        
    return list(event_types), list(argument_roles)

In [ ]:
def process_data(raw_data, buffer, label2id, is_test=False, tasks=[], eval_tasks=[]):
    full_len = []
    sent_total = 0
    data = []
    prompt_len = []
    response_len = []
    none_data = []

    temp_buffer = {}
    b_sent_id_map = {}

    for idx, sample in enumerate(raw_data):
        sent_id_to_sentence = {i: content['sentence'] for i, content in enumerate(sample["content"])}
        sent_id_set = set(sent_id_to_sentence.keys())
        sent_total += len(sent_id_to_sentence)
        sent_to_existing_events = {}

        for event in sample.get("events", []):
            if event.get("type_id", -1) == -1: continue
            event_type = event.get("type")
            description = event.get("description", "")
            
            if label2id[event_type] not in tasks:
                if not (is_test and label2id[event_type] in eval_tasks):
                    continue

            if event_type not in temp_buffer:
                temp_buffer[event_type] = []
            
            for mention in event.get("mention", []):
                sent_id = mention.get("sent_id")
                             
                if sent_id not in sent_to_existing_events:
                    sent_to_existing_events[sent_id] = []
                args = [[arg["text"], arg["role"]] for arg in mention.get("arguments", [])]
                event_info = [mention.get("trigger_word"), event_type, args]
                sent_to_existing_events[sent_id].append(event_info)

                if len(temp_buffer[event_type]) < 10:
                    b_sent_id_map[f"{idx}_{sent_id}"] = event_type

        for sent_id, events in sent_to_existing_events.items():
            sent_txt = sent_id_to_sentence[sent_id]
            sent_id_set.remove(sent_id)

            response = json.dumps({"events": events})
            trigger = [item[0] for item in events]

            data.append({"system_prompt": system_prompt, 
                         "user_prompt": user_prompt.format(text=sent_txt, trigger=trigger), 
                         "sent": sent_txt,
                         "response": response, 
                         "t_system_prompt": t_system_prompt,
                         "t_user_prompt": t_user_prompt.format(text=sent_txt, result=response, trigger=trigger), 
                         })

            if f"{idx}_{sent_id}" in b_sent_id_map:
                temp_buffer[b_sent_id_map[f"{idx}_{sent_id}"]].append(data[-1])


        for sent_id in sent_id_set:
            sent_txt = sent_id_to_sentence[sent_id]

            response = json.dumps({"events": []})
            
            none_data.append({"system_prompt": system_prompt, 
                              "user_prompt": user_prompt.format(text=sent_txt, trigger=[]), 
                              "response": response, 
                              "sent": sent_txt,
                              "t_system_prompt": t_system_prompt,
                              "t_user_prompt": t_user_prompt.format(text=sent_txt, result=response, trigger=[]), 
                              })


    if is_test:
        # data.extend(none_data)
        pass
    else:
        # data.extend(random.sample(list(none_data), min(len(none_data), len(data) // 100)))

        data.extend(buffer)
        
        for k, v in temp_buffer.items():
            buffer.extend(v)

        # for sample in data:
        #     prompt = tokenizer.apply_chat_template(
        #             [{"role": "system", "content": sample['t_system_prompt']},
        #             {"role": "user", "content": sample['t_user_prompt']}],
        #             add_generation_prompt=True,
        #             tokenize=False 
        #         )
        #     full = prompt + sample['response'] + tokenizer.eos_token
            
        #     prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)
        #     full_tokens = tokenizer.encode(full, add_special_tokens=False)
        #     response_tokens = full_tokens[len(prompt_tokens):]

        #     full_len.append(len(full_tokens))
        #     prompt_len.append(len(prompt_tokens))
        #     response_len.append(len(response_tokens))
    
    return data, full_len, prompt_len, response_len

In [26]:
import os

def save(data, data_name, tasks, label2id, buffer, eval_tasks): 
    os.makedirs(f"data/{data_name}", exist_ok=True)
    train, _, _, _ = process_data(data["train"], buffer=buffer, label2id=label2id, tasks=tasks)
    with open(f"data/{data_name}/train.jsonl", "w", encoding="utf-8") as f:
        for item in train:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    dev, _, prompt_len, _ = process_data(data["validation"], [], label2id=label2id, is_test=True, tasks=tasks, eval_tasks=eval_tasks)
    with open(f"data/{data_name}/dev.jsonl", "w", encoding="utf-8") as f:
        for item in dev:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    test, _, prompt_len, _ = process_data(data["test"], [], label2id=label2id, is_test=True, tasks=tasks, eval_tasks=eval_tasks)
    with open(f"data/{data_name}/test.jsonl", "w", encoding="utf-8") as f:
        for item in test:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    return train, dev, test

In [27]:
ace_gen = load_dataset("datht/ace-short-generated-dataset")
geneva_gen = load_dataset("datht/geneva-short-generated-dataset")
maven_gen = load_dataset("datht/maven-short-generated-dataset")
rams_gen = load_dataset("datht/rams-short-generated-dataset")


In [28]:


with open("data/ext-data/ace/label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("data/ext-data/ace/streams.json", "r", encoding="utf-8") as f:
    streams = json.load(f)

buffer = []
eval_tasks = []
for i, tasks in enumerate(streams):
    eval_tasks.extend(tasks)
    save(ace_gen, f"ace_v4/{i}", tasks, label2id, buffer, eval_tasks)

In [23]:


with open("data/ext-data/geneva/label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("data/ext-data/geneva/streams.json", "r", encoding="utf-8") as f:
    streams = json.load(f)

buffer = []
eval_tasks = []
for i, tasks in enumerate(streams):
    eval_tasks.extend(tasks)
    save(geneva_gen, f"geneva_v3/{i}", tasks, label2id, buffer, eval_tasks)

In [9]:

with open("data/ext-data/maven/label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("data/ext-data/maven/streams.json", "r", encoding="utf-8") as f:
    streams = json.load(f)

buffer = []
eval_tasks = []
for i, tasks in enumerate(streams):
    eval_tasks.extend(tasks)
    save(maven_gen, f"maven_v3/{i}", tasks, label2id, buffer, eval_tasks)

In [10]:

with open("data/ext-data/rams/label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("data/ext-data/rams/streams.json", "r", encoding="utf-8") as f:
    streams = json.load(f)

buffer = []
eval_tasks = []
for i, tasks in enumerate(streams):
    eval_tasks.extend(tasks)
    save(rams_gen, f"rams_v3/{i}", tasks, label2id, buffer, eval_tasks)